In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt


from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Thay "HF_TOKEN" bằng đúng cái tên (Label) bạn đã đặt trong Add-ons
secret_label = "HF_TOKEN" 
secret_value = UserSecretsClient().get_secret(secret_label)

# 2. Dùng giá trị secret lấy được để đăng nhập tự động vào Hugging Face
login(token=secret_value, add_to_git_credential=False)

In [2]:
# ============================================================
# 1. Basic imports + config
# ============================================================

import os
import sys
import gc
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import ColQwen2ForRetrieval, ColQwen2Processor
from transformers.utils.import_utils import is_flash_attn_2_available

# -------------------------
# Config
# -------------------------
MODEL_NAME = "vidore/colqwen2-v1.0-hf"
DATASET_NAME = "vidore/docvqa_train"

SEED = 42
VALID_SIZE = 2000
FIXED_PIXELS = 512 * 28 * 28

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Ép dùng 1 GPU nếu có
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())
print("CUDA count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))

Python: 3.12.13
Torch : 2.10.0+cu128
CUDA  : True
CUDA count: 2
GPU 0: Tesla T4


In [3]:
# ============================================================
# 2. Load full dataset and split train/valid
# ============================================================

print("\n========== LOAD FULL DATASET ==========")

full_ds = load_dataset(DATASET_NAME, split="train")

print("Full dataset records:", len(full_ds))
print("Columns:", full_ds.column_names)

# Check required columns
required_cols = {"image", "image_filename", "query"}
missing_cols = required_cols - set(full_ds.column_names)

assert not missing_cols, f"Dataset missing columns: {missing_cols}"

assert len(full_ds) > VALID_SIZE, (
    f"Dataset size must be larger than VALID_SIZE={VALID_SIZE}, "
    f"but got {len(full_ds)}"
)

print("✅ Dataset loaded OK")
print("======================================")


========== LOAD FULL DATASET ==========


README.md:   0%|          | 0.00/516 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

data/train-00000-of-00057.parquet:   0%|          | 0.00/26.8M [00:00<?, ?B/s]

...

data/train-00055-of-00057.parquet:   0%|          | 0.00/186M [00:00<?, ?B/s]

data/train-00056-of-00057.parquet:   0%|          | 0.00/202M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39463 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/51 [00:00<?, ?it/s]

Full dataset records: 39463
Columns: ['image', 'image_filename', 'query', 'answer', 'source']
✅ Dataset loaded OK


In [4]:
# ============================================================
# 3. Split dataset: train / valid
# ============================================================

print("\n========== SPLIT DATASET ==========")

split_ds = full_ds.train_test_split(
    test_size=VALID_SIZE,
    seed=SEED,
    shuffle=True
)

train_ds = split_ds["train"]
valid_ds = split_ds["test"]

print("Train records:", len(train_ds))
print("Valid records:", len(valid_ds))

assert len(valid_ds) == VALID_SIZE, f"Valid size mismatch: {len(valid_ds)}"

print("Train columns:", train_ds.column_names)
print("Valid columns:", valid_ds.column_names)

print("✅ Split OK")
print("===================================")


========== SPLIT DATASET ==========
Train records: 37463
Valid records: 2000
Train columns: ['image', 'image_filename', 'query', 'answer', 'source']
Valid columns: ['image', 'image_filename', 'query', 'answer', 'source']
✅ Split OK


In [5]:
# ============================================================
# 5. Load ColQwen2 model + processor
# ============================================================

print(f"\n========== LOAD MODEL: {MODEL_NAME} ==========")

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    dtype = torch.float16
else:
    device = torch.device("cpu")
    dtype = torch.float32

# Nếu flash_attention_2 lỗi, đổi dòng này thành: attn_impl = "sdpa"
attn_impl = "flash_attention_2" if is_flash_attn_2_available() else "sdpa"

print("Device:", device)
print("Dtype :", dtype)
print("Attention implementation:", attn_impl)


model, loading_info = ColQwen2ForRetrieval.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map=None,              # Quan trọng: không dùng device_map="auto"
    attn_implementation=attn_impl,
    output_loading_info=True,
)

model = model.to(device)
model.eval()

processor = ColQwen2Processor.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    min_pixels=FIXED_PIXELS,
    max_pixels=FIXED_PIXELS
)

print("✅ Model and processor loaded")
print("==============================================")


========== LOAD MODEL: vidore/colqwen2-v1.0-hf ==========
Device: cuda:0
Dtype : torch.float16
Attention implementation: sdpa


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

ColQwen2ForRetrieval LOAD REPORT from: vidore/colqwen2-v1.0-hf
Key                | Status  | 
-------------------+---------+-
vlm.lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

✅ Model and processor loaded


In [6]:
import re
# ============================================================
# Check loading info
# ============================================================

missing = sorted(list(loading_info.get("missing_keys", [])))
unexpected = sorted(list(loading_info.get("unexpected_keys", [])))

ALLOW_MISSING_PATTERNS = [
    r".*position_ids.*",
    r".*rotary_emb\.inv_freq.*",
    r".*inv_freq.*",
    r"vlm\.lm_head\.weight",
]

def is_allowed_missing_key(k):
    return any(re.fullmatch(p, k) for p in ALLOW_MISSING_PATTERNS)

bad_missing = [k for k in missing if not is_allowed_missing_key(k)]

print("\nMissing keys:", len(missing))
print("Unexpected keys:", len(unexpected))
print("Bad missing:", bad_missing)

assert len(bad_missing) == 0, f"❌ Bad missing weights: {bad_missing}"
assert len(unexpected) == 0, f"❌ Unexpected weights: {unexpected}"

# ============================================================
# Check device
# ============================================================

param_devices = sorted({str(p.device) for p in model.parameters()})
print("Parameter devices:", param_devices)

assert param_devices == [str(device)], (
    f"❌ Model is split across devices: {param_devices}"
)

# ============================================================
# Quick valid sample check
# ============================================================

print("\n========== SAMPLE CHECK ==========")

sample = valid_ds[0]

print("Query:", sample["query"])
print("Image filename:", sample["image_filename"])

if "answer" in valid_ds.column_names:
    print("Answer:", sample["answer"])
elif "answers" in valid_ds.column_names:
    print("Answers:", sample["answers"])

print("Image type:", type(sample["image"]))

try:
    print("Image size:", sample["image"].size)
except:
    pass

print("\n✅ DATASET SPLIT + MODEL LOAD OK")


Missing keys: 1
Unexpected keys: 0
Bad missing: []
Parameter devices: ['cuda:0']

========== SAMPLE CHECK ==========
Query: What is the Submitting Office No.?
Provide a short and direct response.
Image filename: 4b58b09cf9f18f9f32b056949d50847e3cbf98e60042b0606617adaf69bb8a00
Answer: 3216.
Image type: <class 'PIL.PngImagePlugin.PngImageFile'>
Image size: (1752, 2246)

✅ DATASET SPLIT + MODEL LOAD OK


In [7]:
# ============================================================
# 6. BUILD CORPUS + QA PAIRS + ENCODE + RETRIEVAL PIPELINE
# Paste below your current successful cell
# ============================================================

import os
import gc
import json
import shutil
import heapq
import numpy as np
import torch

from PIL import Image
from tqdm.auto import tqdm
from datetime import datetime
from collections import OrderedDict


# ============================================================
# 6.1. PIPELINE CONFIG
# ============================================================

IMAGE_BATCH_SIZE = 2
QUERY_BATCH_SIZE = 16
SHARD_SIZE = 100
TOP_K = 10

ARTIFACT_DIR = "/kaggle/working/docvqa_colqwen2_artifacts"
EMB_SHARD_DIR = f"{ARTIFACT_DIR}/corpus_embeddings_shards"
ID_SHARD_DIR = f"{ARTIFACT_DIR}/corpus_ids_shards"
RETRIEVAL_DIR = f"{ARTIFACT_DIR}/retrieval_results"

os.makedirs(ARTIFACT_DIR, exist_ok=True)
os.makedirs(EMB_SHARD_DIR, exist_ok=True)
os.makedirs(ID_SHARD_DIR, exist_ok=True)
os.makedirs(RETRIEVAL_DIR, exist_ok=True)

print("========== PIPELINE CONFIG ==========")
print(f"Dataset       : {DATASET_NAME}")
print(f"Model         : {MODEL_NAME}")
print(f"Device        : {device}")
print(f"Dtype         : {dtype}")
print(f"Fixed pixels  : {FIXED_PIXELS}")
print(f"Image batch   : {IMAGE_BATCH_SIZE}")
print(f"Query batch   : {QUERY_BATCH_SIZE}")
print(f"Shard size    : {SHARD_SIZE}")
print(f"Top K         : {TOP_K}")
print(f"Artifact dir  : {ARTIFACT_DIR}")
print("=====================================")


# ============================================================
# 6.2. COMMON HELPERS
# ============================================================

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_load_image(img):
    """
    Load PIL image an toàn.
    Nếu ảnh lỗi thì trả về ảnh trắng để batch không crash.
    """
    try:
        if img is None:
            raise ValueError("None image")

        if isinstance(img, Image.Image):
            return img.convert("RGB")

        if isinstance(img, str):
            return Image.open(img).convert("RGB")

        if isinstance(img, dict) and "path" in img:
            return Image.open(img["path"]).convert("RGB")

    except Exception as e:
        print("⚠️ Image load error:", repr(e))

    return Image.new("RGB", (224, 224), color=(255, 255, 255))


def move_batch_to_device(batch, device):
    """
    Đưa toàn bộ tensor trong batch sang device hiện tại.
    """
    return {
        k: v.to(device) if hasattr(v, "to") else v
        for k, v in batch.items()
    }


def strip_padding_embeddings(embs, attention_mask):
    """
    embs: [B, L, D]
    attention_mask: [B, L]

    Return:
    - list tensor CPU float32
    - mỗi item shape [valid_L, D]
    """
    mask_cpu = attention_mask.detach().cpu().bool()
    embs_cpu = embs.detach().cpu().float()

    out = []
    for i in range(embs_cpu.shape[0]):
        out.append(embs_cpu[i][mask_cpu[i]])

    return out


def get_answer_from_row(row):
    if "answer" in row and row["answer"] is not None:
        return row["answer"]

    if "answers" in row and row["answers"] is not None:
        return row["answers"]

    return None

========== PIPELINE CONFIG ==========
Dataset       : vidore/docvqa_train
Model         : vidore/colqwen2-v1.0-hf
Device        : cuda:0
Dtype         : torch.float16
Fixed pixels  : 401408
Image batch   : 2
Query batch   : 16
Shard size    : 100
Top K         : 10
Artifact dir  : /kaggle/working/docvqa_colqwen2_artifacts


In [8]:
# ============================================================
# 7. BUILD CORPUS INDEX FROM FULL DATASET
# ============================================================

print("\n========== BUILD CORPUS INDEX ==========")

corpus_id_to_idx = OrderedDict()

for i in tqdm(range(len(full_ds)), desc="Indexing full corpus"):
    doc_id = full_ds[i]["image_filename"]

    if doc_id is None:
        doc_id = f"doc_{i}"

    if doc_id not in corpus_id_to_idx:
        corpus_id_to_idx[doc_id] = i

corpus_ids = list(corpus_id_to_idx.keys())

print("✅ Corpus index built")
print(f"Full records          : {len(full_ds):,}")
print(f"Unique corpus images  : {len(corpus_ids):,}")
print(f"Duplicated records    : {len(full_ds) - len(corpus_ids):,}")


# ============================================================
# 8. BUILD QA PAIRS FROM VALID SET
# ============================================================

print("\n========== BUILD QA PAIRS FROM VALID ==========")

qa_pairs = []

for row in tqdm(valid_ds, desc="Building valid QA pairs"):
    image_filename = row["image_filename"]

    if image_filename in corpus_id_to_idx:
        qa_pairs.append({
            "query": row["query"],
            "image_filename": image_filename,
            "answer": get_answer_from_row(row)
        })

print("✅ QA pairs built")
print(f"Valid records     : {len(valid_ds):,}")
print(f"QA pairs for eval : {len(qa_pairs):,}")

assert len(qa_pairs) == len(valid_ds), (
    f"QA pairs mismatch: got {len(qa_pairs)}, expected {len(valid_ds)}. "
    "Some valid ground-truth images are missing from corpus."
)

print("\nExample corpus id:", corpus_ids[0])
print("Example QA pair  :", qa_pairs[0])


========== BUILD CORPUS INDEX ==========


Indexing full corpus:   0%|          | 0/39463 [00:00<?, ?it/s]

✅ Corpus index built
Full records          : 39,463
Unique corpus images  : 10,189
Duplicated records    : 29,274

========== BUILD QA PAIRS FROM VALID ==========


Building valid QA pairs:   0%|          | 0/2000 [00:00<?, ?it/s]

✅ QA pairs built
Valid records     : 2,000
QA pairs for eval : 2,000

Example corpus id: 78282e984cac0f5ff43f26347ebc6d84fcd6ec20c8d777bdbc0e3983c7b163f2
Example QA pair  : {'query': 'What is the Submitting Office No.?\nProvide a short and direct response.', 'image_filename': '4b58b09cf9f18f9f32b056949d50847e3cbf98e60042b0606617adaf69bb8a00', 'answer': '3216.'}


In [9]:
# ============================================================
# 9. SAVE METADATA
# ============================================================

print("\n========== SAVE METADATA ==========")

with open(f"{ARTIFACT_DIR}/corpus_ids.json", "w", encoding="utf-8") as f:
    json.dump(corpus_ids, f, ensure_ascii=False, indent=2)

with open(f"{ARTIFACT_DIR}/corpus_id_to_idx.json", "w", encoding="utf-8") as f:
    json.dump(dict(corpus_id_to_idx), f, ensure_ascii=False, indent=2)

with open(f"{ARTIFACT_DIR}/qa_pairs.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, ensure_ascii=False, indent=2)

config = {
    "dataset_name": DATASET_NAME,
    "model_name": MODEL_NAME,
    "num_full_records": len(full_ds),
    "num_train_records": len(train_ds),
    "num_valid_records": len(valid_ds),
    "num_unique_corpus_images": len(corpus_ids),
    "num_qa_pairs": len(qa_pairs),
    "split_seed": SEED,
    "valid_size": VALID_SIZE,
    "fixed_pixels": FIXED_PIXELS,
    "image_batch_size": IMAGE_BATCH_SIZE,
    "query_batch_size": QUERY_BATCH_SIZE,
    "shard_size": SHARD_SIZE,
    "top_k": TOP_K,
    "device": str(device),
    "torch_dtype": str(dtype),
    "artifact_dir": ARTIFACT_DIR,
    "created_at": datetime.now().isoformat()
}

with open(f"{ARTIFACT_DIR}/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("✅ Metadata saved:")
print(f"{ARTIFACT_DIR}/corpus_ids.json")
print(f"{ARTIFACT_DIR}/corpus_id_to_idx.json")
print(f"{ARTIFACT_DIR}/qa_pairs.json")
print(f"{ARTIFACT_DIR}/config.json")


========== SAVE METADATA ==========
✅ Metadata saved:
/kaggle/working/docvqa_colqwen2_artifacts/corpus_ids.json
/kaggle/working/docvqa_colqwen2_artifacts/corpus_id_to_idx.json
/kaggle/working/docvqa_colqwen2_artifacts/qa_pairs.json
/kaggle/working/docvqa_colqwen2_artifacts/config.json


In [10]:
# ============================================================
# 10. SHARD HELPERS
# ============================================================

def shard_exists(shard_idx):
    emb_path = f"{EMB_SHARD_DIR}/shard_{shard_idx:05d}.npy"
    ids_path = f"{ID_SHARD_DIR}/shard_{shard_idx:05d}.json"

    return os.path.exists(emb_path) and os.path.exists(ids_path)


def save_manifest(manifest, filename="manifest_partial.json"):
    path = f"{ARTIFACT_DIR}/{filename}"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    return path


def save_shard(shard_idx, shard_embeddings, shard_ids):
    """
    Save một shard:
    - embeddings: .npy object array
    - ids: .json

    Vì ColQwen2 là multi-vector embedding, mỗi ảnh có thể có shape khác nhau.
    """
    emb_path = f"{EMB_SHARD_DIR}/shard_{shard_idx:05d}.npy"
    ids_path = f"{ID_SHARD_DIR}/shard_{shard_idx:05d}.json"

    np.save(
        emb_path,
        np.array(shard_embeddings, dtype=object),
        allow_pickle=True
    )

    with open(ids_path, "w", encoding="utf-8") as f:
        json.dump(shard_ids, f, ensure_ascii=False, indent=2)

    return {
        "shard_idx": shard_idx,
        "num_items": len(shard_ids),
        "embedding_path": emb_path,
        "ids_path": ids_path,
        "first_shape": list(shard_embeddings[0].shape) if len(shard_embeddings) > 0 else None,
        "saved_as": "object_array"
    }


# ============================================================
# 11. ENCODE IMAGES
# ============================================================

def encode_images_as_numpy_list(images, batch_size=1):
    """
    Encode list PIL images bằng ColQwen2ForRetrieval.

    Return:
    - list np.ndarray
    - mỗi item shape [valid_L, D]
    """
    all_embeddings = []

    for start in range(0, len(images), batch_size):
        batch_imgs = images[start:start + batch_size]

        inputs = processor(
            images=batch_imgs,
            return_tensors="pt"
        )

        inputs = move_batch_to_device(inputs, device)

        with torch.inference_mode():
            outputs = model(**inputs)
            embs = outputs.embeddings

        batch_embeddings = strip_padding_embeddings(
            embs,
            inputs["attention_mask"]
        )

        for emb in batch_embeddings:
            all_embeddings.append(emb.numpy().astype(np.float32))

        del inputs, outputs, embs, batch_embeddings
        clear_memory()

    return all_embeddings


def embed_corpus_save_shards(
    corpus_ids,
    corpus_id_to_idx,
    dataset,
    batch_size=1,
    shard_size=100,
    resume=True
):
    """
    Embed toàn bộ corpus images và save theo shard.

    Output:
    - corpus_embeddings_shards/shard_xxxxx.npy
    - corpus_ids_shards/shard_xxxxx.json
    - manifest_partial.json
    - manifest.json
    """

    model.eval()

    num_docs = len(corpus_ids)
    num_shards = int(np.ceil(num_docs / shard_size))

    manifest = {
        "dataset_name": DATASET_NAME,
        "model_name": MODEL_NAME,
        "num_docs": num_docs,
        "num_shards": num_shards,
        "batch_size": batch_size,
        "shard_size": shard_size,
        "fixed_pixels": FIXED_PIXELS,
        "device": str(device),
        "torch_dtype": str(dtype),
        "created_at": datetime.now().isoformat(),
        "shards": []
    }

    print("\n========== START EMBEDDING CORPUS ==========")
    print(f"Corpus size : {num_docs:,}")
    print(f"Num shards  : {num_shards}")
    print(f"Batch size  : {batch_size}")
    print(f"Shard size  : {shard_size}")
    print(f"Save dir    : {ARTIFACT_DIR}")
    print("============================================")

    for shard_idx in range(num_shards):
        shard_start = shard_idx * shard_size
        shard_end = min((shard_idx + 1) * shard_size, num_docs)

        current_ids = corpus_ids[shard_start:shard_end]

        if resume and shard_exists(shard_idx):
            print(f"\n⏭️ Skip shard {shard_idx:05d} vì đã tồn tại.")

            shard_info = {
                "shard_idx": shard_idx,
                "num_items": len(current_ids),
                "embedding_path": f"{EMB_SHARD_DIR}/shard_{shard_idx:05d}.npy",
                "ids_path": f"{ID_SHARD_DIR}/shard_{shard_idx:05d}.json",
                "status": "skipped_existing"
            }

            manifest["shards"].append(shard_info)
            save_manifest(manifest, "manifest_partial.json")
            continue

        print(f"\n🚀 Embedding shard {shard_idx:05d}: images {shard_start} → {shard_end - 1}")

        shard_embeddings = []
        shard_ids = []

        for start in tqdm(
            range(0, len(current_ids), batch_size),
            desc=f"Shard {shard_idx:05d}"
        ):
            batch_ids = current_ids[start:start + batch_size]

            batch_imgs = [
                safe_load_image(dataset[corpus_id_to_idx[doc_id]]["image"])
                for doc_id in batch_ids
            ]

            batch_embeddings = encode_images_as_numpy_list(
                batch_imgs,
                batch_size=batch_size
            )

            for doc_id, emb_np in zip(batch_ids, batch_embeddings):
                shard_embeddings.append(emb_np)
                shard_ids.append(doc_id)

            del batch_imgs, batch_embeddings
            clear_memory()

        shard_info = save_shard(
            shard_idx=shard_idx,
            shard_embeddings=shard_embeddings,
            shard_ids=shard_ids
        )

        shard_info["status"] = "saved"
        manifest["shards"].append(shard_info)

        print(f"✅ Saved shard {shard_idx:05d}")
        print(f"   Items: {len(shard_ids)}")
        print(f"   Path : {shard_info['embedding_path']}")

        del shard_embeddings, shard_ids
        clear_memory()

        save_manifest(manifest, "manifest_partial.json")

    manifest["finished_at"] = datetime.now().isoformat()
    manifest["status"] = "finished"

    save_manifest(manifest, "manifest.json")

    print("\n🎉 DONE embedding corpus")
    print(f"✅ Manifest saved: {ARTIFACT_DIR}/manifest.json")

    return manifest

In [11]:
# ============================================================
# 12. RUN CORPUS EMBEDDING
# ============================================================

manifest = embed_corpus_save_shards(
    corpus_ids=corpus_ids,
    corpus_id_to_idx=corpus_id_to_idx,
    dataset=full_ds,
    batch_size=IMAGE_BATCH_SIZE,
    shard_size=SHARD_SIZE,
    resume=True
)


========== START EMBEDDING CORPUS ==========
Corpus size : 10,189
Num shards  : 102
Batch size  : 2
Shard size  : 100
Save dir    : /kaggle/working/docvqa_colqwen2_artifacts

🚀 Embedding shard 00000: images 0 → 99


Shard 00000:   0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved shard 00000
   Items: 100
   Path : /kaggle/working/docvqa_colqwen2_artifacts/corpus_embeddings_shards/shard_00000.npy

🚀 Embedding shard 00001: images 100 → 199


Shard 00001:   0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved shard 00001
   Items: 100
   Path : /kaggle/working/docvqa_colqwen2_artifacts/corpus_embeddings_shards/shard_00001.npy

🚀 Embedding shard 00002: images 200 → 299


...

✅ Saved shard 00099
   Items: 100
   Path : /kaggle/working/docvqa_colqwen2_artifacts/corpus_embeddings_shards/shard_00099.npy

🚀 Embedding shard 00100: images 10000 → 10099


Shard 00100:   0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved shard 00100
   Items: 100
   Path : /kaggle/working/docvqa_colqwen2_artifacts/corpus_embeddings_shards/shard_00100.npy

🚀 Embedding shard 00101: images 10100 → 10188


Shard 00101:   0%|          | 0/45 [00:00<?, ?it/s]

✅ Saved shard 00101
   Items: 89
   Path : /kaggle/working/docvqa_colqwen2_artifacts/corpus_embeddings_shards/shard_00101.npy

🎉 DONE embedding corpus
✅ Manifest saved: /kaggle/working/docvqa_colqwen2_artifacts/manifest.json


In [12]:
# ============================================================
# 13. CHECK SAVED CORPUS SHARDS
# ============================================================

print("\n========== CHECK CORPUS SHARDS ==========")

with open(f"{ARTIFACT_DIR}/manifest.json", "r", encoding="utf-8") as f:
    manifest_check = json.load(f)

print("Num docs   :", manifest_check["num_docs"])
print("Num shards :", manifest_check["num_shards"])
print("Status     :", manifest_check["status"])

if len(manifest_check["shards"]) > 0:
    first_shard = manifest_check["shards"][0]

    first_emb = np.load(first_shard["embedding_path"], allow_pickle=True)

    with open(first_shard["ids_path"], "r", encoding="utf-8") as f:
        first_ids = json.load(f)

    print("First shard embedding length:", len(first_emb))
    print("First shard ids length      :", len(first_ids))
    print("First id                    :", first_ids[0])
    print("First embedding shape       :", first_emb[0].shape)
    print("First embedding dtype       :", first_emb[0].dtype)

print("=========================================")


# ============================================================
# 14. ENCODE VALID QUERIES
# ============================================================

QUERY_EMB_PATH = f"{ARTIFACT_DIR}/query_embeddings.npy"
QUERY_META_PATH = f"{ARTIFACT_DIR}/query_metadata.json"
QUERY_CONFIG_PATH = f"{ARTIFACT_DIR}/query_config.json"

with open(f"{ARTIFACT_DIR}/qa_pairs.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

queries = [item["query"] for item in qa_pairs]

print("\n========== QUERY SET ==========")
print("Số QA pairs:", len(qa_pairs))
print("Số queries :", len(queries))
print("Ví dụ query:", queries[0])
print("===============================")


def encode_queries_as_numpy_list(texts, batch_size=16):
    """
    Encode queries bằng ColQwen2ForRetrieval.

    Return:
    - list np.ndarray
    - mỗi item shape [valid_L, D]
    """
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = processor(
            text=batch_texts,
            return_tensors="pt",
            padding=True
        )

        inputs = move_batch_to_device(inputs, device)

        with torch.inference_mode():
            outputs = model(**inputs)
            embs = outputs.embeddings

        batch_embeddings = strip_padding_embeddings(
            embs,
            inputs["attention_mask"]
        )

        for emb in batch_embeddings:
            all_embeddings.append(emb.numpy().astype(np.float32))

        del inputs, outputs, embs, batch_embeddings
        clear_memory()

    return all_embeddings


def embed_queries_save(
    queries,
    qa_pairs,
    batch_size=16,
    resume=True
):
    """
    Encode valid queries và save lại.

    Output:
    - query_embeddings.npy
    - query_metadata.json
    - query_config.json
    """
    if resume and os.path.exists(QUERY_EMB_PATH) and os.path.exists(QUERY_META_PATH):
        print("✅ Query embeddings đã tồn tại, load lại.")

        query_embeddings = np.load(QUERY_EMB_PATH, allow_pickle=True)

        with open(QUERY_META_PATH, "r", encoding="utf-8") as f:
            query_metadata = json.load(f)

        print("Loaded query embeddings:", len(query_embeddings))
        print("Loaded query metadata  :", len(query_metadata))
        print("Shape mẫu:", query_embeddings[0].shape)

        return query_embeddings, query_metadata

    model.eval()

    all_query_embeddings = []
    query_metadata = []

    for start in tqdm(
        range(0, len(queries), batch_size),
        desc="Embedding valid queries"
    ):
        batch_queries = queries[start:start + batch_size]
        batch_meta = qa_pairs[start:start + batch_size]

        batch_embeddings = encode_queries_as_numpy_list(
            batch_queries,
            batch_size=batch_size
        )

        for emb_np, meta in zip(batch_embeddings, batch_meta):
            all_query_embeddings.append(emb_np)

            query_metadata.append({
                "query": meta["query"],
                "ground_truth_image_filename": meta["image_filename"],
                "answer": meta["answer"]
            })

        del batch_embeddings
        clear_memory()

    np.save(
        QUERY_EMB_PATH,
        np.array(all_query_embeddings, dtype=object),
        allow_pickle=True
    )

    with open(QUERY_META_PATH, "w", encoding="utf-8") as f:
        json.dump(query_metadata, f, ensure_ascii=False, indent=2)

    query_config = {
        "num_queries": len(all_query_embeddings),
        "batch_size_query": batch_size,
        "saved_path": QUERY_EMB_PATH,
        "metadata_path": QUERY_META_PATH,
        "fixed_pixels": FIXED_PIXELS,
        "created_at": datetime.now().isoformat()
    }

    with open(QUERY_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(query_config, f, ensure_ascii=False, indent=2)

    print("\n✅ Đã save query embeddings")
    print("Embeddings:", QUERY_EMB_PATH)
    print("Metadata  :", QUERY_META_PATH)
    print("Config    :", QUERY_CONFIG_PATH)
    print("Num query :", len(all_query_embeddings))
    print("Shape mẫu :", all_query_embeddings[0].shape)

    return all_query_embeddings, query_metadata


query_embeddings, query_metadata = embed_queries_save(
    queries=queries,
    qa_pairs=qa_pairs,
    batch_size=QUERY_BATCH_SIZE,
    resume=True
)


========== CHECK CORPUS SHARDS ==========
Num docs   : 10189
Num shards : 102
Status     : finished
First shard embedding length: 100
First shard ids length      : 100
First id                    : 78282e984cac0f5ff43f26347ebc6d84fcd6ec20c8d777bdbc0e3983c7b163f2
First embedding shape       : (491, 128)
First embedding dtype       : float32

========== QUERY SET ==========
Số QA pairs: 2000
Số queries : 2000
Ví dụ query: What is the Submitting Office No.?
Provide a short and direct response.


Embedding valid queries:   0%|          | 0/125 [00:00<?, ?it/s]


✅ Đã save query embeddings
Embeddings: /kaggle/working/docvqa_colqwen2_artifacts/query_embeddings.npy
Metadata  : /kaggle/working/docvqa_colqwen2_artifacts/query_metadata.json
Config    : /kaggle/working/docvqa_colqwen2_artifacts/query_config.json
Num query : 2000
Shape mẫu : (28, 128)


In [13]:
# ============================================================
# 15. CHECK QUERY EMBEDDINGS
# ============================================================

print("\n========== CHECK QUERY EMBEDDINGS ==========")

query_embs = np.load(QUERY_EMB_PATH, allow_pickle=True)

with open(QUERY_META_PATH, "r", encoding="utf-8") as f:
    query_meta = json.load(f)

print("Số query embeddings:", len(query_embs))
print("Số query metadata  :", len(query_meta))

for i in range(min(3, len(query_embs))):
    print("-" * 60)
    print("Query:", query_meta[i]["query"])
    print("GT image:", query_meta[i]["ground_truth_image_filename"])
    print("Answer:", query_meta[i]["answer"])
    print("Embedding shape:", query_embs[i].shape)
    print("Dtype:", query_embs[i].dtype)

print("============================================")


========== CHECK QUERY EMBEDDINGS ==========
Số query embeddings: 2000
Số query metadata  : 2000
------------------------------------------------------------
Query: What is the Submitting Office No.?
Provide a short and direct response.
GT image: 4b58b09cf9f18f9f32b056949d50847e3cbf98e60042b0606617adaf69bb8a00
Answer: 3216.
Embedding shape: (28, 128)
Dtype: float32
------------------------------------------------------------
Query: What is the muscle protein breakdown % of total body breakdown in young men?
Keep it short and to the point.
GT image: 49921ac56b38f971e4bb0d5e11b0c42aa46d1b6bffbfdc0ec31e7fca73cf9808
Answer: 70%.
Embedding shape: (35, 128)
Dtype: float32
------------------------------------------------------------
Query: What is the Fax No of Stephen M. Parker?
Provide a succinct answer.
GT image: b568f75eed313508381c3c1d5d94a5170dee99f4f677952432c303871eaaa94d
Answer: 609-497-2304.
Embedding shape: (28, 128)
Dtype: float32


In [14]:
# ============================================================
# 16. E0 DENSE RETRIEVAL MAXSIM
# ============================================================

E0_RESULTS_PATH = f"{RETRIEVAL_DIR}/e0_dense_topk_results.json"
E0_METRICS_PATH = f"{RETRIEVAL_DIR}/e0_dense_metrics.json"
E0_PARTIAL_PATH = f"{RETRIEVAL_DIR}/e0_dense_topk_results_partial.json"

query_embeddings = np.load(QUERY_EMB_PATH, allow_pickle=True)

with open(QUERY_META_PATH, "r", encoding="utf-8") as f:
    query_metadata = json.load(f)

with open(f"{ARTIFACT_DIR}/manifest.json", "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("\n========== LOAD RETRIEVAL INPUT ==========")
print("Loaded query embeddings:", len(query_embeddings))
print("Loaded query metadata  :", len(query_metadata))
print("Loaded shards          :", manifest["num_shards"])
print("==========================================")


def maxsim_score(query_emb, image_emb):
    """
    query_emb: [Q, D]
    image_emb: [P, D]

    Score = sum over query tokens of max dot product over image patches.
    """
    query_emb = np.asarray(query_emb, dtype=np.float32)
    image_emb = np.asarray(image_emb, dtype=np.float32)

    sim = query_emb @ image_emb.T
    max_per_query_token = sim.max(axis=1)
    score = max_per_query_token.sum()

    return float(score)


def compute_retrieval_metrics(results, k_values=[1, 5, 10]):
    metrics = {}
    n = len(results)

    for k in k_values:
        hit = 0

        for item in results:
            gt = item["ground_truth_image_filename"]
            pred_ids = [x["image_id"] for x in item["topk"][:k]]

            if gt in pred_ids:
                hit += 1

        metrics[f"Recall@{k}"] = hit / n

    reciprocal_ranks = []

    for item in results:
        gt = item["ground_truth_image_filename"]
        pred_ids = [x["image_id"] for x in item["topk"][:10]]

        rr = 0.0

        for rank, pred_id in enumerate(pred_ids, start=1):
            if pred_id == gt:
                rr = 1.0 / rank
                break

        reciprocal_ranks.append(rr)

    metrics["MRR@10"] = float(np.mean(reciprocal_ranks))

    ndcg_scores = []

    for item in results:
        gt = item["ground_truth_image_filename"]
        pred_ids = [x["image_id"] for x in item["topk"][:10]]

        ndcg = 0.0

        for rank, pred_id in enumerate(pred_ids, start=1):
            if pred_id == gt:
                ndcg = 1.0 / np.log2(rank + 1)
                break

        ndcg_scores.append(ndcg)

    metrics["NDCG@10"] = float(np.mean(ndcg_scores))

    return metrics


def run_e0_dense_retrieval_by_shards(
    query_embeddings,
    query_metadata,
    manifest,
    top_k=10,
    resume=True
):
    """
    E0 dense retrieval bằng MaxSim.

    Cách này duyệt theo shard:
    - Load mỗi shard một lần
    - So sánh shard đó với tất cả query
    - Giữ top_k bằng heap

    Nhanh và đỡ I/O hơn kiểu mỗi query load toàn bộ shard.
    """
    if resume and os.path.exists(E0_RESULTS_PATH) and os.path.exists(E0_METRICS_PATH):
        print("✅ E0 results đã tồn tại, load lại.")

        with open(E0_RESULTS_PATH, "r", encoding="utf-8") as f:
            results = json.load(f)

        with open(E0_METRICS_PATH, "r", encoding="utf-8") as f:
            metrics_info = json.load(f)

        print("Loaded E0 metrics:", metrics_info["metrics"])

        return results, metrics_info

    num_queries = len(query_embeddings)

    # Mỗi query giữ một min-heap: [(score, image_id), ...]
    top_heaps = [[] for _ in range(num_queries)]

    print("\n========== START E0 DENSE RETRIEVAL ==========")
    print(f"Queries : {num_queries:,}")
    print(f"Shards  : {manifest['num_shards']}")
    print(f"Top K   : {top_k}")
    print("==============================================")

    for shard in tqdm(manifest["shards"], desc="Scanning corpus shards"):
        emb_path = shard["embedding_path"]
        ids_path = shard["ids_path"]

        shard_embs = np.load(emb_path, allow_pickle=True)

        with open(ids_path, "r", encoding="utf-8") as f:
            shard_ids = json.load(f)

        for q_idx, query_emb in enumerate(query_embeddings):
            heap = top_heaps[q_idx]

            for image_id, image_emb in zip(shard_ids, shard_embs):
                score = maxsim_score(query_emb, image_emb)

                if len(heap) < top_k:
                    heapq.heappush(heap, (score, image_id))
                else:
                    if score > heap[0][0]:
                        heapq.heapreplace(heap, (score, image_id))

        del shard_embs, shard_ids
        clear_memory()

        # Save partial sau mỗi shard
        partial_heaps = []
        for heap in top_heaps:
            sorted_items = sorted(heap, key=lambda x: x[0], reverse=True)
            partial_heaps.append([
                {"image_id": image_id, "score": float(score)}
                for score, image_id in sorted_items
            ])

        with open(E0_PARTIAL_PATH, "w", encoding="utf-8") as f:
            json.dump(partial_heaps, f, ensure_ascii=False, indent=2)

    results = []

    for i in range(num_queries):
        meta = query_metadata[i]

        sorted_topk = sorted(
            top_heaps[i],
            key=lambda x: x[0],
            reverse=True
        )

        result_item = {
            "query_index": i,
            "query": meta["query"],
            "ground_truth_image_filename": meta["ground_truth_image_filename"],
            "answer": meta["answer"],
            "topk": [
                {
                    "rank": rank + 1,
                    "image_id": image_id,
                    "score": float(score)
                }
                for rank, (score, image_id) in enumerate(sorted_topk)
            ]
        }

        results.append(result_item)

    metrics = compute_retrieval_metrics(
        results,
        k_values=[1, 5, 10]
    )

    metrics_info = {
        "experiment": "E0_ColQwen2_MaxSim_Dense_Retrieval",
        "num_queries": len(results),
        "num_corpus_images": manifest["num_docs"],
        "num_shards": manifest["num_shards"],
        "top_k": top_k,
        "fixed_pixels": FIXED_PIXELS,
        "created_at": datetime.now().isoformat(),
        "metrics": metrics
    }

    with open(E0_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    with open(E0_METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(metrics_info, f, ensure_ascii=False, indent=2)

    print("\n✅ E0 Dense Retrieval finished")
    print("Results:", E0_RESULTS_PATH)
    print("Metrics:", E0_METRICS_PATH)
    print("Metrics:", metrics)

    return results, metrics_info


e0_results, e0_metrics = run_e0_dense_retrieval_by_shards(
    query_embeddings=query_embeddings,
    query_metadata=query_metadata,
    manifest=manifest,
    top_k=TOP_K,
    resume=True
)


# ============================================================
# 17. CHECK E0 RESULTS
# ============================================================

print("\n========== CHECK E0 RESULTS ==========")

with open(E0_METRICS_PATH, "r", encoding="utf-8") as f:
    e0_metrics_check = json.load(f)

print(json.dumps(e0_metrics_check, indent=2, ensure_ascii=False))

with open(E0_RESULTS_PATH, "r", encoding="utf-8") as f:
    e0_results_check = json.load(f)

example = e0_results_check[0]

print("\nExample result:")
print("Query:", example["query"])
print("GT   :", example["ground_truth_image_filename"])
print("TopK :")

for item in example["topk"][:5]:
    marker = "✅" if item["image_id"] == example["ground_truth_image_filename"] else "  "
    print(
        f"{item['rank']:>2}. {marker} {item['image_id']} | score={item['score']:.4f}"
    )

print("======================================")


========== LOAD RETRIEVAL INPUT ==========
Loaded query embeddings: 2000
Loaded query metadata  : 2000
Loaded shards          : 102

========== START E0 DENSE RETRIEVAL ==========
Queries : 2,000
Shards  : 102
Top K   : 10


Scanning corpus shards:   0%|          | 0/102 [00:00<?, ?it/s]


✅ E0 Dense Retrieval finished
Results: /kaggle/working/docvqa_colqwen2_artifacts/retrieval_results/e0_dense_topk_results.json
Metrics: /kaggle/working/docvqa_colqwen2_artifacts/retrieval_results/e0_dense_metrics.json
Metrics: {'Recall@1': 0.3775, 'Recall@5': 0.563, 'Recall@10': 0.6295, 'MRR@10': 0.45798531746031745, 'NDCG@10': 0.4991823934197356}

========== CHECK E0 RESULTS ==========
{
  "experiment": "E0_ColQwen2_MaxSim_Dense_Retrieval",
  "num_queries": 2000,
  "num_corpus_images": 10189,
  "num_shards": 102,
  "top_k": 10,
  "fixed_pixels": 401408,
  "created_at": "2026-06-11T22:40:49.764995",
  "metrics": {
    "Recall@1": 0.3775,
    "Recall@5": 0.563,
    "Recall@10": 0.6295,
    "MRR@10": 0.45798531746031745,
    "NDCG@10": 0.4991823934197356
  }
}

Example result:
Query: What is the Submitting Office No.?
Provide a short and direct response.
GT   : 4b58b09cf9f18f9f32b056949d50847e3cbf98e60042b0606617adaf69bb8a00
TopK :
 1.    b241d26b1be7f5049a7c2ac415aa5265312e0f6dc0aef954f

In [15]:
# ============================================================
# 18. ZIP ARTIFACTS
# ============================================================

print("\n========== ZIP ARTIFACTS ==========")

zip_base = "/kaggle/working/docvqa_colqwen2_artifacts"
zip_path = f"{zip_base}.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/kaggle/working",
    base_dir="docvqa_colqwen2_artifacts"
)

print("✅ Zipped artifact:")
print(zip_path)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

with open(f"{ARTIFACT_DIR}/manifest.json", "r", encoding="utf-8") as f:
    final_manifest = json.load(f)

print("\n========== FINAL SUMMARY ==========")
print(f"Artifact folder : {ARTIFACT_DIR}")
print(f"Zip file        : {zip_path}")
print(f"Corpus images   : {final_manifest['num_docs']:,}")
print(f"QA pairs        : {len(query_metadata):,}")
print(f"Num shards      : {final_manifest['num_shards']}")
print("Saved files:")
print(" - corpus_ids.json")
print(" - corpus_id_to_idx.json")
print(" - qa_pairs.json")
print(" - config.json")
print(" - manifest.json")
print(" - manifest_partial.json")
print(" - corpus_embeddings_shards/*.npy")
print(" - corpus_ids_shards/*.json")
print(" - query_embeddings.npy")
print(" - query_metadata.json")
print(" - query_config.json")
print(" - retrieval_results/e0_dense_topk_results.json")
print(" - retrieval_results/e0_dense_metrics.json")
print(" - retrieval_results/e0_dense_topk_results_partial.json")
print("===================================")


========== ZIP ARTIFACTS ==========
✅ Zipped artifact:
/kaggle/working/docvqa_colqwen2_artifacts.zip

========== FINAL SUMMARY ==========
Artifact folder : /kaggle/working/docvqa_colqwen2_artifacts
Zip file        : /kaggle/working/docvqa_colqwen2_artifacts.zip
Corpus images   : 10,189
QA pairs        : 2,000
Num shards      : 102
Saved files:
 - corpus_ids.json
 - corpus_id_to_idx.json
 - qa_pairs.json
 - config.json
 - manifest.json
 - manifest_partial.json
 - corpus_embeddings_shards/*.npy
 - corpus_ids_shards/*.json
 - query_embeddings.npy
 - query_metadata.json
 - query_config.json
 - retrieval_results/e0_dense_topk_results.json
 - retrieval_results/e0_dense_metrics.json
 - retrieval_results/e0_dense_topk_results_partial.json
